# 06 — Checkpointed model-family screening

This notebook compares the nine frozen model families on the two feature pipelines retained separately for each split seed and target by Notebook 05.

Every candidate is refitted inside the saved inner folds. This notebook does **not** tune hyperparameters, inspect outer-test outcomes, or estimate final performance. It retains two feature-pipeline/model combinations per seed and target for focused tuning in Notebook 07.


## 1. Dependency check

Confirm that the three external boosting libraries required by the frozen nine-family scope are available. This cell never installs packages automatically.


In [1]:
import importlib.util
import os

# Avoid noisy macOS physical-core detection in joblib while preserving the
# available logical-core limit for parallel estimators.
os.environ.setdefault(
    "LOKY_MAX_CPU_COUNT",
    str(max(1, (os.cpu_count() or 2) - 1)),
)

required_external_packages = ["xgboost", "lightgbm", "catboost"]
missing_packages = [
    package
    for package in required_external_packages
    if importlib.util.find_spec(package) is None
]
if missing_packages:
    raise ModuleNotFoundError(
        "Install the missing packages in this notebook environment, restart the "
        f"kernel, and rerun: {missing_packages}"
    )
print("External model packages are available.")


External model packages are available.


Load the numerical, statistical, pipeline, and model-family tools.


In [2]:
from pathlib import Path
import hashlib
import json
import platform
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from sklearn import __version__ as sklearn_version
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier, __version__ as xgboost_version
from lightgbm import LGBMClassifier, __version__ as lightgbm_version
from catboost import CatBoostClassifier, __version__ as catboost_version


Locate the final-run inputs and create the matching result directory.


In [3]:
working_directory = Path.cwd().resolve()
project_root = next(
    (
        folder
        for folder in [working_directory, *working_directory.parents]
        if (folder / "AGENTS.md").is_file()
        and (folder / "docs/research_protocol.md").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Launch this notebook from within the project directory.")

final_run_directory = project_root / "results/final_pipeline/06_final_fit_and_performance_summary"
aggregation_directory = final_run_directory / "02_patient_level_aggregation"
split_directory = final_run_directory / "03_setup_and_splits"
feature_screen_directory = final_run_directory / "05_feature_pipeline_screening"
output_directory = final_run_directory / "06_model_family_screening"
output_directory.mkdir(parents=True, exist_ok=True)

input_paths = {
    "predictors": aggregation_directory / "primary_1040_26_predictors.csv",
    "outcomes": aggregation_directory / "primary_1040_outcome_metadata.csv",
    "feature_manifest": split_directory / "primary_feature_manifest.csv",
    "outer_splits": split_directory / "outer_split_assignments.csv",
    "inner_folds": split_directory / "inner_fold_assignments.csv",
    "retained_pipelines": feature_screen_directory / "retained_feature_pipelines.csv",
    "engineering_manifest": feature_screen_directory / "engineering_configuration_manifest.csv",
}
missing_inputs = [name for name, file_path in input_paths.items() if not file_path.is_file()]
assert not missing_inputs, f"Missing required inputs: {missing_inputs}"

print("Project root:", project_root)
print("Output directory:", output_directory.relative_to(project_root))


Project root: /Users/rafsan_temp/Library/CloudStorage/OneDrive-SeattleUniversity/SU Projects/pd-fall-risk
Output directory: results/final_pipeline/06_final_fit_and_performance_summary/06_model_family_screening


## 2. Load the final input, splits, and local candidate queue

Read the single 26-feature table and verify that Notebook 05 supplied exactly two distinct feature pipelines for every seed and target.


In [4]:
predictors = pd.read_csv(
    input_paths["predictors"],
    dtype={"PATNO": "string"},
    low_memory=False,
).set_index("PATNO")
outcome_metadata = pd.read_csv(
    input_paths["outcomes"],
    dtype={"PATNO": "string"},
    usecols=["PATNO", "falls_class"],
)
feature_manifest = pd.read_csv(input_paths["feature_manifest"])
outer_assignments = pd.read_csv(
    input_paths["outer_splits"],
    dtype={"PATNO": "string"},
)
inner_assignments = pd.read_csv(
    input_paths["inner_folds"],
    dtype={"PATNO": "string"},
)
retained_pipelines = pd.read_csv(input_paths["retained_pipelines"])
engineering_manifest = pd.read_csv(input_paths["engineering_manifest"])

features = feature_manifest["feature"].tolist()
outcomes = outcome_metadata.set_index("PATNO")["falls_class"].astype(int)
targets = ["direct", "stage_1", "stage_2"]
split_seeds = sorted(outer_assignments["split_seed"].unique())

queue_counts = retained_pipelines.groupby(["split_seed", "target"]).size()
queue_distinct = retained_pipelines.groupby(
    ["split_seed", "target"]
)["pipeline_key"].nunique()

assert predictors.columns.tolist() == features and len(features) == 26
assert predictors.shape == (1040, 26)
assert set(predictors.index) == set(outcomes.index)
assert len(queue_counts) == 60 and queue_counts.eq(2).all()
assert queue_distinct.eq(2).all()
assert set(retained_pipelines["target"]) == set(targets)

print("Patients:", len(predictors))
print("Local feature-pipeline candidates:", len(retained_pipelines))
print(retained_pipelines.groupby("target").size().to_string())


Patients: 1040
Local feature-pipeline candidates: 120
target
direct     40
stage_1    40
stage_2    40


## 3. Frozen model-family scope

Use one prespecified starting configuration per family. These settings screen families efficiently; focused hyperparameter tuning occurs only after this notebook retains two combinations locally.


In [5]:
MODEL_FAMILIES = [
    "logistic",
    "linear_svc",
    "rbf_svc",
    "random_forest",
    "extra_trees",
    "hist_gradient_boosting",
    "xgboost",
    "lightgbm",
    "catboost",
]
MODEL_ORDER = {family: order for order, family in enumerate(MODEL_FAMILIES)}
MODEL_BACKEND = {
    "logistic": "scaled_dense",
    "linear_svc": "scaled_dense",
    "rbf_svc": "scaled_dense",
    "random_forest": "unscaled_dense",
    "extra_trees": "unscaled_dense",
    "hist_gradient_boosting": "native",
    "xgboost": "native",
    "lightgbm": "native",
    "catboost": "native",
}
MODEL_STARTING_CONFIG = {
    "logistic": {"C": 1.0, "class_weight": "balanced"},
    "linear_svc": {"C": 1.0, "class_weight": "balanced"},
    "rbf_svc": {"C": 1.0, "gamma": "scale", "class_weight": "balanced"},
    "random_forest": {
        "n_estimators": 300, "max_depth": 6,
        "min_samples_leaf": 5, "class_weight": "balanced",
    },
    "extra_trees": {
        "n_estimators": 300, "max_depth": 6,
        "min_samples_leaf": 5, "class_weight": "balanced",
    },
    "hist_gradient_boosting": {
        "max_leaf_nodes": 15, "learning_rate": 0.05,
        "l2_regularization": 1.0, "class_weight": "balanced",
    },
    "xgboost": {
        "n_estimators": 300, "max_depth": 2, "learning_rate": 0.08,
        "subsample": 0.8, "colsample_bytree": 0.8,
        "reg_lambda": 5.0, "sample_weight": "balanced",
    },
    "lightgbm": {
        "n_estimators": 300, "num_leaves": 15, "learning_rate": 0.05,
        "min_child_samples": 20, "subsample": 0.8, "subsample_freq": 1,
        "colsample_bytree": 0.8, "reg_lambda": 1.0,
        "class_weight": "balanced",
    },
    "catboost": {
        "iterations": 300, "depth": 4, "learning_rate": 0.05,
        "l2_leaf_reg": 5.0, "auto_class_weights": "Balanced",
    },
}

model_scope = pd.DataFrame([
    {
        "model_family": family,
        "backend": MODEL_BACKEND[family],
        "screening_parameters": json.dumps(
            MODEL_STARTING_CONFIG[family],
            sort_keys=True,
        ),
        "candidate_winner": True,
    }
    for family in MODEL_FAMILIES
] + [{
    "model_family": "dummy_prior",
    "backend": "none",
    "screening_parameters": '{"strategy": "prior"}',
    "candidate_winner": False,
}])

display(model_scope)


,model_family,backend,screening_parameters,candidate_winner
0,logistic,scaled_dense,"{""C"": 1.0, ""class_weight"": ""balanced""}",True
1,linear_svc,scaled_dense,"{""C"": 1.0, ""class_weight"": ""balanced""}",True
2,rbf_svc,scaled_dense,"{""C"": 1.0, ""class_weight"": ""balanced"", ""gamma""...",True
3,random_forest,unscaled_dense,"{""class_weight"": ""balanced"", ""max_depth"": 6, ""...",True
4,extra_trees,unscaled_dense,"{""class_weight"": ""balanced"", ""max_depth"": 6, ""...",True
5,hist_gradient_boosting,native,"{""class_weight"": ""balanced"", ""l2_regularizatio...",True
6,xgboost,native,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0...",True
7,lightgbm,native,"{""class_weight"": ""balanced"", ""colsample_bytree...",True
8,catboost,native,"{""auto_class_weights"": ""Balanced"", ""depth"": 4,...",True
9,dummy_prior,none,"{""strategy"": ""prior""}",False


## 4. Clinical preprocessing definitions

Declare the final nominal, ordinal, and missing-state groups. The primary 1,040-patient cohort does not receive a no-history indicator.


In [6]:
NOMINAL_FEATURES = {
    "DXPOSINS", "DXRIGID", "DOPTHERST", "FEATPOSHYP",
    "ANYFAMPD", "DXTREMOR", "DXBRADY", "DOMSIDE",
}
ORDINAL_FEATURES = {
    "FRZGT12M", "SCAU14", "SCAU16", "NP1SLPD", "NP1URIN",
    "NP3GAIT_COMBINED_MAX", "NP3PSTBL_COMBINED_MAX", "NHY_COMBINED_MAX",
    "NP1CNST",
}
MISSING_INDICATORS = [
    "FOG_FORM_MISSING", "NQ_FORM_MISSING", "PART_IV_FORM_MISSING",
]


def representation_group(source):
    if source in {"FRZGT12M", "FOG_FORM_MISSING"}:
        return "GROUP_FREEZING_FORM"
    if source in {"NQ_GAUSSIAN_REVISION", "NQ_FORM_MISSING"}:
        return "GROUP_NEUROQOL_FORM"
    if source in {"NP4TOT", "PART_IV_FORM_MISSING"}:
        return "GROUP_PART_IV_FORM"
    return source


def clinical_frame(frame):
    raw = frame[features].copy()
    indicators = pd.DataFrame(index=raw.index)
    indicators["FOG_FORM_MISSING"] = raw["FRZGT12M"].isna().astype(int)
    indicators["NQ_FORM_MISSING"] = raw["NQ_GAUSSIAN_REVISION"].isna().astype(int)
    indicators["PART_IV_FORM_MISSING"] = raw["NP4TOT"].isna().astype(int)

    structural_zero = raw["NP4TOT"].isna() & raw["DOPTHERST"].eq("No")
    raw.loc[structural_zero, "NP4TOT"] = 0.0
    return pd.concat([raw, indicators], axis=1)


## 5. Fold-specific statistical selector

Recreate corrected FDR screening using only the current inner-training fold. Nominal missingness remains an explicit category and declared representation groups are kept together.


In [7]:
def bh_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan)
    valid = np.flatnonzero(np.isfinite(p_values))
    if not len(valid):
        return adjusted
    order = valid[np.argsort(p_values[valid])]
    ranked = p_values[order] * len(valid) / np.arange(1, len(valid) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted[order] = np.minimum(ranked, 1.0)
    return adjusted


def numeric_p_value(values, target, ordinal):
    observed = pd.DataFrame({"value": values, "target": target}).dropna()
    groups = [
        observed.loc[observed["target"].eq(label), "value"].astype(float).to_numpy()
        for label in sorted(observed["target"].unique())
    ]
    if len(groups) < 2 or min(len(group) for group in groups) < 2:
        return np.nan
    if np.unique(np.concatenate(groups)).size < 2:
        return 1.0

    if ordinal:
        test = (
            stats.mannwhitneyu(*groups, alternative="two-sided")
            if len(groups) == 2
            else stats.kruskal(*groups)
        )
        return float(test.pvalue)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        normality = [
            stats.normaltest(group).pvalue
            if len(group) >= 8 and np.unique(group).size >= 3
            else 0.0
            for group in groups
        ]
        variance_p = stats.levene(*groups, center="median").pvalue
    parametric = (
        all(np.isfinite(normality))
        and min(normality) >= 0.05
        and np.isfinite(variance_p)
        and variance_p >= 0.05
    )
    if len(groups) == 2:
        test = (
            stats.ttest_ind(*groups, equal_var=True)
            if parametric
            else stats.mannwhitneyu(*groups, alternative="two-sided")
        )
    else:
        test = stats.f_oneway(*groups) if parametric else stats.kruskal(*groups)
    return float(test.pvalue)


def categorical_p_value(values, target, random_seed):
    categories = values.astype("string").fillna("Missing")
    table = pd.crosstab(categories, target)
    table = table.loc[table.sum(axis=1).gt(0)]
    if table.shape[0] < 2 or table.shape[1] < 2:
        return 1.0
    asymptotic = stats.chi2_contingency(table, correction=False)
    sparse = (
        asymptotic.expected_freq.min() < 1
        or (asymptotic.expected_freq < 5).mean() > 0.20
    )
    if not sparse:
        return float(asymptotic.pvalue)
    method = stats.PermutationMethod(
        n_resamples=999,
        rng=np.random.default_rng(random_seed),
    )
    return float(
        stats.chi2_contingency(
            table,
            correction=False,
            method=method,
        ).pvalue
    )


def select_groups_fdr(raw, target, random_seed):
    p_values = []
    for position, column in enumerate(raw.columns):
        if column in NOMINAL_FEATURES or column in MISSING_INDICATORS:
            p_value = categorical_p_value(
                raw[column],
                target,
                random_seed + position,
            )
        else:
            p_value = numeric_p_value(
                raw[column],
                target,
                ordinal=column in ORDINAL_FEATURES,
            )
        p_values.append(p_value)

    q_values = bh_adjust(p_values)
    selected_sources = [
        column
        for column, q_value in zip(raw.columns, q_values)
        if np.isfinite(q_value) and q_value <= 0.05
    ]
    if not selected_sources:
        finite = np.flatnonzero(np.isfinite(q_values))
        fallback = finite[np.argmin(q_values[finite])] if len(finite) else 0
        selected_sources = [raw.columns[fallback]]
    return {representation_group(source) for source in selected_sources}


## 6. Complete fold-fitted candidate transformer

Fit clinical missing-data handling, encoding, scaling, optional engineering, and supervised selection inside each training fold. Compatible boosting families receive native numeric missing values only after selection has been fitted on the training fold.


In [8]:
engineering_definitions = {
    row.branch: {
        "kind": row.branch_kind,
        "sources": row.source_features.split(" | "),
        "threshold": None if pd.isna(row.variance_threshold) else float(row.variance_threshold),
    }
    for row in engineering_manifest.itertuples(index=False)
}


class CandidateTransformer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        candidate_kind,
        selector,
        selector_parameter,
        branch=None,
        backend="scaled_dense",
        random_state=42,
    ):
        self.candidate_kind = candidate_kind
        self.selector = selector
        self.selector_parameter = selector_parameter
        self.branch = branch
        self.backend = backend
        self.random_state = random_state

    def _fit_preprocessing(self, raw):
        self.nominal_columns_ = [column for column in raw if column in NOMINAL_FEATURES]
        self.numeric_columns_ = [column for column in raw if column not in self.nominal_columns_]

        freezing_mode = raw["FRZGT12M"].mode(dropna=True)
        if freezing_mode.empty:
            raise ValueError("FRZGT12M has no observed training value.")
        self.freezing_fill_ = float(freezing_mode.iloc[0])

        median_columns = [column for column in self.numeric_columns_ if column != "FRZGT12M"]
        self.medians_ = raw[median_columns].median()
        if self.medians_.isna().any():
            missing = self.medians_[self.medians_.isna()].index.tolist()
            raise ValueError(f"No training value available for: {missing}")

        dense_numeric = raw[self.numeric_columns_].copy()
        dense_numeric["FRZGT12M"] = dense_numeric["FRZGT12M"].fillna(
            self.freezing_fill_
        )
        dense_numeric[median_columns] = dense_numeric[median_columns].fillna(
            self.medians_
        )
        self.scaler_ = StandardScaler().fit(dense_numeric)

        nominal = (
            raw[self.nominal_columns_]
            .astype("string")
            .fillna("Missing")
            .astype(str)
        )
        self.encoder_ = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ).fit(nominal)
        self.encoded_nominal_columns_ = self.encoder_.get_feature_names_out(
            self.nominal_columns_
        ).tolist()

        source_by_column = {column: column for column in self.numeric_columns_}
        encoded_sources = [
            source
            for source, categories in zip(
                self.nominal_columns_,
                self.encoder_.categories_,
            )
            for _ in categories
        ]
        source_by_column.update(
            dict(zip(self.encoded_nominal_columns_, encoded_sources))
        )
        self.group_by_column_ = {
            column: representation_group(source)
            for column, source in source_by_column.items()
        }

    def _matrices(self, raw):
        dense_numeric = raw[self.numeric_columns_].copy()
        dense_numeric["FRZGT12M"] = dense_numeric["FRZGT12M"].fillna(
            self.freezing_fill_
        )
        median_columns = [column for column in self.numeric_columns_ if column != "FRZGT12M"]
        dense_numeric[median_columns] = dense_numeric[median_columns].fillna(
            self.medians_
        )
        scaled_numeric = pd.DataFrame(
            self.scaler_.transform(dense_numeric),
            index=raw.index,
            columns=self.numeric_columns_,
        )

        nominal = (
            raw[self.nominal_columns_]
            .astype("string")
            .fillna("Missing")
            .astype(str)
        )
        encoded = pd.DataFrame(
            self.encoder_.transform(nominal),
            index=raw.index,
            columns=self.encoded_nominal_columns_,
        )
        native_numeric = raw[self.numeric_columns_].apply(
            pd.to_numeric,
            errors="coerce",
        )
        return {
            "scaled_dense": pd.concat([scaled_numeric, encoded], axis=1),
            "unscaled_dense": pd.concat([dense_numeric, encoded], axis=1),
            "native": pd.concat([native_numeric, encoded], axis=1),
        }

    def _apply_engineering(self, matrix, fit):
        if self.candidate_kind != "engineered_representation":
            return matrix
        definition = engineering_definitions[self.branch]
        matrix = matrix.copy()
        sources = definition["sources"]

        if definition["kind"] == "interaction":
            matrix[self.branch] = (
                matrix[sources[0]].to_numpy()
                * matrix[sources[1]].to_numpy()
            )
            return matrix

        if fit:
            self.pca_ = PCA(
                n_components=definition["threshold"],
                svd_solver="full",
            ).fit(matrix[sources])
        components = self.pca_.transform(matrix[sources])
        matrix = matrix.drop(columns=sources)
        names = [
            f"{self.branch}_PC{index + 1}"
            for index in range(components.shape[1])
        ]
        matrix[names] = components
        return matrix

    def _select_columns(self, raw, dense, target):
        if self.selector == "none":
            self.selected_groups_ = set(self.group_by_column_.values())
            return dense.columns.tolist()

        if self.selector == "corrected_fdr":
            selected_groups = select_groups_fdr(raw, target, self.random_state)
        elif self.selector == "l1":
            C_value = float(self.selector_parameter.split("=")[1])
            base_estimator = LogisticRegression(
                solver="liblinear",
                l1_ratio=1.0,
                C=C_value,
                class_weight="balanced",
                max_iter=5000,
                random_state=self.random_state,
            )
            selector = OneVsRestClassifier(base_estimator, n_jobs=-1)
            selector.fit(dense, target)
            importance = np.max(
                np.vstack([
                    np.abs(estimator.coef_).reshape(-1)
                    for estimator in selector.estimators_
                ]),
                axis=0,
            )
            selected_groups = {
                self.group_by_column_[column]
                for column, value in zip(dense.columns, importance)
                if value > 1e-10
            }
            if not selected_groups:
                strongest = dense.columns[int(np.argmax(importance))]
                selected_groups = {self.group_by_column_[strongest]}
        else:
            multiplier = 1.25 if "1.25" in self.selector_parameter else 1.0
            selector = ExtraTreesClassifier(
                n_estimators=120,
                max_depth=6,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=self.random_state,
                n_jobs=-1,
            )
            selector.fit(dense, target)
            group_importance = {}
            for column, value in zip(dense.columns, selector.feature_importances_):
                group = self.group_by_column_[column]
                group_importance[group] = group_importance.get(group, 0.0) + float(value)
            threshold = np.median(list(group_importance.values())) * multiplier
            selected_groups = {
                group
                for group, value in group_importance.items()
                if value >= threshold
            }
            if not selected_groups:
                selected_groups = {max(group_importance, key=group_importance.get)}

        self.selected_groups_ = selected_groups
        return [
            column
            for column in dense
            if self.group_by_column_[column] in selected_groups
        ]

    def fit(self, X, y):
        raw = clinical_frame(X)
        self._fit_preprocessing(raw)
        matrices = self._matrices(raw)
        dense = self._apply_engineering(matrices["scaled_dense"], fit=True)

        if self.candidate_kind == "engineered_representation":
            self.force_dense_ = True
            self.selected_columns_ = dense.columns.tolist()
            self.selected_groups_ = set(self.group_by_column_.values()) | {self.branch}
        else:
            self.force_dense_ = False
            self.selected_columns_ = self._select_columns(raw, dense, y)
        if not self.selected_columns_:
            raise RuntimeError("Candidate transformer selected no columns.")
        return self

    def transform(self, X):
        raw = clinical_frame(X)
        matrices = self._matrices(raw)
        if self.force_dense_:
            output = self._apply_engineering(
                matrices["scaled_dense"],
                fit=False,
            )
        else:
            output = matrices[self.backend]
        return output[self.selected_columns_]

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.selected_columns_, dtype=object)


## 7. Models, targets, and metrics

Construct deterministic starting models and the three target-specific datasets. Stage 2 includes only true fallers and codes rare fall as class 0.


In [9]:
def build_model(family, target_name, random_seed):
    parameters = dict(MODEL_STARTING_CONFIG[family])
    if family == "logistic":
        return LogisticRegression(
            max_iter=5000,
            random_state=random_seed,
            **parameters,
        ), None
    if family == "linear_svc":
        return LinearSVC(
            dual="auto",
            max_iter=10000,
            random_state=random_seed,
            **parameters,
        ), None
    if family == "rbf_svc":
        return SVC(cache_size=1000, random_state=random_seed, **parameters), None
    if family == "random_forest":
        return RandomForestClassifier(
            random_state=random_seed,
            n_jobs=-1,
            **parameters,
        ), None
    if family == "extra_trees":
        return ExtraTreesClassifier(
            random_state=random_seed,
            n_jobs=-1,
            **parameters,
        ), None
    if family == "hist_gradient_boosting":
        return HistGradientBoostingClassifier(
            random_state=random_seed,
            **parameters,
        ), None
    if family == "xgboost":
        weight_mode = parameters.pop("sample_weight")
        classes = 3 if target_name == "direct" else 2
        parameters.update({
            "objective": "multi:softprob" if classes == 3 else "binary:logistic",
            "eval_metric": "mlogloss" if classes == 3 else "logloss",
            "random_state": random_seed,
            "n_jobs": -1,
            "verbosity": 0,
        })
        if classes == 3:
            parameters["num_class"] = 3
        return XGBClassifier(**parameters), weight_mode
    if family == "lightgbm":
        return LGBMClassifier(
            random_state=random_seed,
            n_jobs=-1,
            verbosity=-1,
            **parameters,
        ), None

    loss = "MultiClass" if target_name == "direct" else "Logloss"
    return CatBoostClassifier(
        random_seed=random_seed,
        loss_function=loss,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
        **parameters,
    ), None


def target_data(target_name, training_ids, validation_ids):
    y_train = outcomes.loc[list(training_ids)]
    y_validation = outcomes.loc[list(validation_ids)]
    if target_name == "direct":
        return list(training_ids), list(validation_ids), y_train, y_validation
    if target_name == "stage_1":
        return (
            list(training_ids),
            list(validation_ids),
            y_train.gt(0).astype(int),
            y_validation.gt(0).astype(int),
        )
    y_train = y_train[y_train.gt(0)]
    y_validation = y_validation[y_validation.gt(0)]
    return (
        y_train.index.tolist(),
        y_validation.index.tolist(),
        y_train.eq(2).astype(int),
        y_validation.eq(2).astype(int),
    )


def classification_metrics(target_name, truth, predictions):
    labels = [0, 1, 2] if target_name == "direct" else [0, 1]
    recalls = recall_score(
        truth,
        predictions,
        labels=labels,
        average=None,
        zero_division=0,
    )
    result = {
        "macro_f1": f1_score(truth, predictions, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(truth, predictions),
        "accuracy": accuracy_score(truth, predictions),
        "recall_class_0": recalls[0],
        "recall_class_1": recalls[1],
        "recall_class_2": recalls[2] if target_name == "direct" else np.nan,
    }
    result["priority_recall"] = (
        result["recall_class_0"]
        if target_name == "stage_2"
        else result["recall_class_1"]
    )
    return result


Capture model warnings during fitting. A real convergence or library warning stops the current unit with one concise error instead of filling the notebook with repeated warnings.


In [10]:
def fit_checked(estimator, X, y, context, **fit_parameters):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        estimator.fit(X, y, **fit_parameters)

    if caught:
        unique_messages = list(dict.fromkeys(str(item.message) for item in caught))
        convergence = any(
            issubclass(item.category, ConvergenceWarning)
            for item in caught
        )
        warning_type = "convergence warning" if convergence else "model warning"
        raise RuntimeError(
            f"{context} emitted a {warning_type}: {unique_messages[0]}"
        )
    return estimator


## 8. Reproducibility and checkpoint identity

Hash every upstream input and the complete model scope. Existing checkpoints are reused only when this identity matches.


In [11]:
RUN_VERSION = "final-notebook-06-nine-family-screen-v2-26-features-d28"


def file_digest(file_path):
    digest = hashlib.sha256()
    with open(file_path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


identity = {
    "run_version": RUN_VERSION,
    "input_sha256": {
        name: file_digest(file_path)
        for name, file_path in input_paths.items()
    },
    "model_scope": model_scope.to_dict("records"),
    "targets": targets,
    "inner_folds": 5,
    "selection_metric": "mean inner-fold macro F1",
    "near_tie_margin": 0.01,
}
configuration_hash = hashlib.sha256(
    json.dumps(identity, sort_keys=True).encode()
).hexdigest()
manifest_path = output_directory / "run_manifest.json"
manifest = {
    **identity,
    "configuration_hash": configuration_hash,
    "python": platform.python_version(),
    "scikit_learn": sklearn_version,
    "xgboost": xgboost_version,
    "lightgbm": lightgbm_version,
    "catboost": catboost_version,
}
if manifest_path.is_file():
    existing = json.loads(manifest_path.read_text())
    if existing.get("configuration_hash") != configuration_hash:
        raise RuntimeError(
            "Existing checkpoints belong to a different configuration. "
            "Preserve them and use a new output directory."
        )
else:
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")

print("Configuration hash:", configuration_hash[:16])
print(
    pd.Series({
        name: manifest[name]
        for name in [
            "python", "scikit_learn", "xgboost", "lightgbm", "catboost",
        ]
    }).to_string()
)


Configuration hash: be25e329310c7324
python          3.13.12
scikit_learn      1.8.0
xgboost           3.2.0
lightgbm          4.7.0
catboost         1.2.10


## 9. Work estimate

The long-running cell contains 60 resumable units. Each unit evaluates two local feature pipelines across nine families and five inner folds, plus a separate dummy benchmark.


In [12]:
queue_sizes = retained_pipelines.groupby(["split_seed", "target"]).size()
planned_model_fits = int((queue_sizes * len(MODEL_FAMILIES) * 5).sum())
planned_dummy_fits = len(queue_sizes) * 5
work_plan = pd.DataFrame({
    "work_units": [len(queue_sizes)],
    "candidate_model_fits": [planned_model_fits],
    "dummy_benchmark_fits": [planned_dummy_fits],
    "checkpoint_boundary": ["one split seed × one target"],
})
display(work_plan)
print("A stopped run resumes from the next fully completed unit.")


,work_units,candidate_model_fits,dummy_benchmark_fits,checkpoint_boundary
0,60,5400,300,one split seed × one target


A stopped run resumes from the next fully completed unit.


## 10. Run the checkpointed family screen

This is the long-running cell. It reports progress, elapsed time, and a session-based ETA after every completed seed/target unit.


In [13]:
def atomic_csv(frame, file_path):
    temporary = file_path.with_suffix(file_path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(file_path)


checkpoint_path = output_directory / "family_screen_checkpoint.csv"
dummy_checkpoint_path = output_directory / "dummy_benchmark_checkpoint.csv"

saved = pd.read_csv(checkpoint_path) if checkpoint_path.is_file() else pd.DataFrame()
saved_dummy = (
    pd.read_csv(dummy_checkpoint_path)
    if dummy_checkpoint_path.is_file()
    else pd.DataFrame()
)
complete_units = set()
if not saved.empty:
    for key, group in saved.groupby(["split_seed", "target"]):
        expected = int(queue_sizes.loc[key]) * len(MODEL_FAMILIES) * 5
        if len(group) == expected:
            complete_units.add(key)

rows = (
    []
    if saved.empty
    else saved[
        saved.apply(
            lambda row: (row["split_seed"], row["target"]) in complete_units,
            axis=1,
        )
    ].to_dict("records")
)
dummy_rows = (
    []
    if saved_dummy.empty
    else saved_dummy[
        saved_dummy.apply(
            lambda row: (row["split_seed"], row["target"]) in complete_units,
            axis=1,
        )
    ].to_dict("records")
)

session_durations = []
total_units = len(queue_sizes)
print(
    f"Resuming with {len(complete_units)}/{total_units} units complete.",
    flush=True,
)

for split_seed in split_seeds:
    seed_inner = inner_assignments.loc[
        inner_assignments["split_seed"].eq(split_seed)
    ]
    outer_train_ids = set(
        outer_assignments.loc[
            outer_assignments["split_seed"].eq(split_seed)
            & outer_assignments["role"].eq("train"),
            "PATNO",
        ]
    )
    outer_test_ids = set(
        outer_assignments.loc[
            outer_assignments["split_seed"].eq(split_seed)
            & outer_assignments["role"].eq("test"),
            "PATNO",
        ]
    )
    assert set(seed_inner["PATNO"]) == outer_train_ids
    assert outer_test_ids.isdisjoint(outer_train_ids)

    for target_name in targets:
        unit = (split_seed, target_name)
        if unit in complete_units:
            continue

        unit_start = time.perf_counter()
        unit_rows = []
        unit_dummy_rows = []
        candidate_rows = retained_pipelines.loc[
            retained_pipelines["split_seed"].eq(split_seed)
            & retained_pipelines["target"].eq(target_name)
        ]
        print(
            f"Starting seed {split_seed:02d}, {target_name}: "
            f"{len(candidate_rows)} pipelines × 9 families × 5 folds",
            flush=True,
        )

        for inner_fold in range(5):
            validation_ids = sorted(
                seed_inner.loc[
                    seed_inner["inner_validation_fold"].eq(inner_fold),
                    "PATNO",
                ].tolist()
            )
            training_ids = sorted(outer_train_ids - set(validation_ids))
            assert set(training_ids).isdisjoint(validation_ids)
            assert outer_test_ids.isdisjoint([*training_ids, *validation_ids])

            train_ids, valid_ids, y_train, y_validation = target_data(
                target_name,
                training_ids,
                validation_ids,
            )
            random_seed = 300_000 + split_seed * 10 + inner_fold

            dummy = DummyClassifier(strategy="prior")
            dummy.fit(np.zeros((len(y_train), 1)), y_train)
            dummy_predictions = dummy.predict(np.zeros((len(y_validation), 1)))
            unit_dummy_rows.append({
                "split_seed": split_seed,
                "target": target_name,
                "inner_validation_fold": inner_fold,
                "training_patients": len(y_train),
                "validation_patients": len(y_validation),
                **classification_metrics(
                    target_name,
                    y_validation,
                    dummy_predictions,
                ),
            })

            for candidate_row in candidate_rows.itertuples(index=False):
                branch = None if pd.isna(candidate_row.branch) else candidate_row.branch
                for family in MODEL_FAMILIES:
                    transformer = CandidateTransformer(
                        candidate_kind=candidate_row.candidate_kind,
                        selector=candidate_row.selector,
                        selector_parameter=candidate_row.selector_parameter,
                        branch=branch,
                        backend=MODEL_BACKEND[family],
                        random_state=random_seed,
                    )
                    estimator, weight_mode = build_model(
                        family,
                        target_name,
                        random_seed,
                    )
                    pipeline = Pipeline([
                        ("candidate", transformer),
                        ("model", estimator),
                    ])
                    fit_parameters = {}
                    if weight_mode == "balanced":
                        fit_parameters["model__sample_weight"] = compute_sample_weight(
                            "balanced",
                            y_train,
                        )

                    fit_start = time.perf_counter()
                    fit_checked(
                        pipeline,
                        predictors.loc[train_ids],
                        y_train,
                        context=(
                            f"seed {split_seed}, {target_name}, fold {inner_fold}, "
                            f"{candidate_row.pipeline_key}, {family}"
                        ),
                        **fit_parameters,
                    )
                    predictions = pipeline.predict(predictors.loc[valid_ids])
                    fitted_transformer = pipeline.named_steps["candidate"]

                    unit_rows.append({
                        "split_seed": split_seed,
                        "target": target_name,
                        "inner_validation_fold": inner_fold,
                        "retained_feature_rank": candidate_row.retained_rank,
                        "configuration_id": candidate_row.configuration_id,
                        "pipeline_key": candidate_row.pipeline_key,
                        "candidate_kind": candidate_row.candidate_kind,
                        "selector": candidate_row.selector,
                        "selector_parameter": candidate_row.selector_parameter,
                        "branch": branch,
                        "branch_kind": candidate_row.branch_kind,
                        "model_family": family,
                        "backend": (
                            "scaled_dense"
                            if fitted_transformer.force_dense_
                            else MODEL_BACKEND[family]
                        ),
                        "training_patients": len(y_train),
                        "validation_patients": len(y_validation),
                        "selected_columns": len(fitted_transformer.selected_columns_),
                        "selected_groups": len(fitted_transformer.selected_groups_),
                        "selected_column_names": " | ".join(
                            fitted_transformer.selected_columns_
                        ),
                        "fit_seconds": time.perf_counter() - fit_start,
                        **classification_metrics(
                            target_name,
                            y_validation,
                            predictions,
                        ),
                    })

        expected_rows = len(candidate_rows) * len(MODEL_FAMILIES) * 5
        assert len(unit_rows) == expected_rows
        assert len(unit_dummy_rows) == 5
        rows.extend(unit_rows)
        dummy_rows.extend(unit_dummy_rows)
        complete_units.add(unit)

        family_checkpoint = pd.DataFrame(rows).sort_values(
            [
                "split_seed", "target", "inner_validation_fold",
                "pipeline_key", "model_family",
            ]
        )
        dummy_checkpoint = pd.DataFrame(dummy_rows).sort_values(
            ["split_seed", "target", "inner_validation_fold"]
        )
        atomic_csv(family_checkpoint, checkpoint_path)
        atomic_csv(dummy_checkpoint, dummy_checkpoint_path)

        duration = time.perf_counter() - unit_start
        session_durations.append(duration)
        remaining = total_units - len(complete_units)
        eta_minutes = remaining * np.mean(session_durations) / 60
        print(
            f"Completed {len(complete_units)}/{total_units} units in "
            f"{duration / 60:.1f} min; estimated remaining {eta_minutes:.1f} min",
            flush=True,
        )

family_folds = pd.DataFrame(rows).sort_values(
    ["split_seed", "target", "inner_validation_fold", "pipeline_key", "model_family"]
).reset_index(drop=True)
dummy_folds = pd.DataFrame(dummy_rows).sort_values(
    ["split_seed", "target", "inner_validation_fold"]
).reset_index(drop=True)
print(f"Family screen complete: {len(family_folds):,} candidate-fold results.")


Resuming with 0/60 units complete.
Starting seed 00, direct: 2 pipelines × 9 families × 5 folds
Completed 1/60 units in 0.6 min; estimated remaining 36.9 min
Starting seed 00, stage_1: 2 pipelines × 9 families × 5 folds
Completed 2/60 units in 0.3 min; estimated remaining 26.9 min
Starting seed 00, stage_2: 2 pipelines × 9 families × 5 folds
Completed 3/60 units in 0.2 min; estimated remaining 21.4 min
Starting seed 01, direct: 2 pipelines × 9 families × 5 folds
Completed 4/60 units in 0.8 min; estimated remaining 26.4 min
Starting seed 01, stage_1: 2 pipelines × 9 families × 5 folds
Completed 5/60 units in 0.3 min; estimated remaining 24.4 min
Starting seed 01, stage_2: 2 pipelines × 9 families × 5 folds
Completed 6/60 units in 0.3 min; estimated remaining 22.6 min
Starting seed 02, direct: 2 pipelines × 9 families × 5 folds
Completed 7/60 units in 0.7 min; estimated remaining 23.9 min
Starting seed 02, stage_1: 2 pipelines × 9 families × 5 folds
Completed 8/60 units in 0.5 min; estim

## 11. Summarize and retain two combinations

Average the five inner folds and apply the locked macro-F1, target-priority-recall, parsimony, and model-order rule within each seed and target.


In [14]:
family_summary = (
    family_folds.groupby(
        [
            "split_seed", "target", "configuration_id", "pipeline_key",
            "candidate_kind", "selector", "selector_parameter",
            "branch", "branch_kind", "model_family", "backend",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        mean_macro_f1=("macro_f1", "mean"),
        sd_macro_f1=("macro_f1", "std"),
        mean_priority_recall=("priority_recall", "mean"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_selected_columns=("selected_columns", "mean"),
        mean_selected_groups=("selected_groups", "mean"),
        mean_fit_seconds=("fit_seconds", "mean"),
        inner_folds=("inner_validation_fold", "nunique"),
    )
)


def retain_two_combinations(group):
    remaining = group.copy()
    retained = []
    for rank in [1, 2]:
        best_macro_f1 = remaining["mean_macro_f1"].max()
        near_ties = remaining.loc[
            remaining["mean_macro_f1"].ge(best_macro_f1 - 0.01)
        ].copy()
        near_ties["model_order"] = near_ties["model_family"].map(MODEL_ORDER)
        chosen = near_ties.sort_values(
            [
                "mean_priority_recall", "mean_selected_groups",
                "mean_selected_columns", "model_order",
                "pipeline_key", "model_family",
            ],
            ascending=[False, True, True, True, True, True],
        ).iloc[0]
        chosen_row = chosen.drop(labels="model_order").to_dict()
        chosen_row["retained_rank"] = rank
        retained.append(chosen_row)
        remaining = remaining.loc[
            ~(
                remaining["pipeline_key"].eq(chosen["pipeline_key"])
                & remaining["model_family"].eq(chosen["model_family"])
            )
        ]
    return pd.DataFrame(retained)


retained_combinations = pd.concat(
    [
        retain_two_combinations(group)
        for _, group in family_summary.groupby(
            ["split_seed", "target"],
            sort=False,
        )
    ],
    ignore_index=True,
).sort_values(["split_seed", "target", "retained_rank"])

family_frequency = (
    retained_combinations.groupby(["target", "model_family"])
    .size()
    .rename("retained_count")
    .reset_index()
    .sort_values(["target", "retained_count"], ascending=[True, False])
)
dummy_summary = (
    dummy_folds.groupby(["split_seed", "target"], as_index=False)
    .agg(
        mean_macro_f1=("macro_f1", "mean"),
        mean_priority_recall=("priority_recall", "mean"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        mean_accuracy=("accuracy", "mean"),
    )
)

display(family_frequency)
print("Retained combination rows:", len(retained_combinations))


,target,model_family,retained_count
3,direct,random_forest,18
0,direct,catboost,15
4,direct,rbf_svc,4
1,direct,extra_trees,2
2,direct,linear_svc,1
9,stage_1,logistic,10
10,stage_1,random_forest,8
11,stage_1,rbf_svc,8
6,stage_1,extra_trees,7
5,stage_1,catboost,4


Retained combination rows: 120


## 12. Validate the model-family screen

Verify complete paired coverage, two retained combinations per seed and target, benchmark separation, checkpoint identity, and the absence of outer-test results.


In [15]:
validation_rows = []


def check(name, condition, detail):
    validation_rows.append({
        "check": name,
        "passed": bool(condition),
        "detail": detail,
    })


unit_counts = family_folds.groupby(["split_seed", "target"]).size()
combination_folds = family_folds.groupby(
    ["split_seed", "target", "pipeline_key", "model_family"]
)["inner_validation_fold"].nunique()
retained_counts = retained_combinations.groupby(["split_seed", "target"]).size()
metric_columns = ["macro_f1", "priority_recall", "balanced_accuracy", "accuracy"]

check(
    "60 target-specific units complete",
    len(unit_counts) == 60 and unit_counts.eq(90).all(),
    "2 pipelines × 9 families × 5 folds",
)
check(
    "all nine model families evaluated",
    set(family_folds["model_family"]) == set(MODEL_FAMILIES),
    "frozen family scope",
)
check(
    "five folds per pipeline and family",
    combination_folds.eq(5).all(),
    "paired inner validation",
)
check(
    "all fitted pipelines retained columns",
    family_folds["selected_columns"].gt(0).all(),
    "no empty model input",
)
check(
    "all metrics are bounded",
    family_folds[metric_columns].apply(
        lambda column: column.between(0, 1).all()
    ).all(),
    "candidate-fold results",
)
check(
    "two combinations retained per seed and target",
    len(retained_counts) == 60 and retained_counts.eq(2).all(),
    "120 focused-tuning rows",
)
check(
    "dummy benchmark has five folds per unit",
    dummy_folds.groupby(["split_seed", "target"])[
        "inner_validation_fold"
    ].nunique().eq(5).all(),
    "benchmark only",
)
check(
    "dummy cannot enter retained combinations",
    "dummy_prior" not in set(retained_combinations["model_family"]),
    "candidate_winner is false",
)
check(
    "no outer-test result generated",
    not any(
        "outer_prediction" in column or "outer_metric" in column
        for column in family_folds.columns
    ),
    "inner validation only",
)
check(
    "checkpoint identity matches",
    json.loads(manifest_path.read_text())["configuration_hash"]
    == configuration_hash,
    "input and model-scope hash",
)
check(
    "no patient-level transformed matrix saved",
    True,
    "only fold metrics and configuration metadata are written",
)

validation = pd.DataFrame(validation_rows)
display(validation)
assert validation["passed"].all(), validation.loc[~validation["passed"]]
print(f"Validation checks passed: {validation['passed'].sum()}/{len(validation)}")


,check,passed,detail
0,60 target-specific units complete,True,2 pipelines × 9 families × 5 folds
1,all nine model families evaluated,True,frozen family scope
2,five folds per pipeline and family,True,paired inner validation
3,all fitted pipelines retained columns,True,no empty model input
4,all metrics are bounded,True,candidate-fold results
5,two combinations retained per seed and target,True,120 focused-tuning rows
6,dummy benchmark has five folds per unit,True,benchmark only
7,dummy cannot enter retained combinations,True,candidate_winner is false
8,no outer-test result generated,True,inner validation only
9,checkpoint identity matches,True,input and model-scope hash


Validation checks passed: 11/11


## 13. Save reusable screening artifacts

Write the completed fold results, summaries, retained tuning queue, benchmark, model scope, and validation table. Differing existing files are never overwritten silently.


In [16]:
def frames_equivalent(current, existing):
    if current.columns.tolist() != existing.columns.tolist() or current.shape != existing.shape:
        return False
    for column in current.columns:
        left = current[column]
        right = existing[column]
        left_numeric = pd.to_numeric(left, errors="coerce")
        right_numeric = pd.to_numeric(right, errors="coerce")
        left_numeric_ok = left_numeric.notna().eq(left.notna()).all()
        right_numeric_ok = right_numeric.notna().eq(right.notna()).all()
        if left_numeric_ok and right_numeric_ok:
            if not np.allclose(
                left_numeric.to_numpy(dtype=float),
                right_numeric.to_numpy(dtype=float),
                rtol=1e-12,
                atol=1e-12,
                equal_nan=True,
            ):
                return False
        else:
            left_text = left.astype("string").fillna("<NA>").reset_index(drop=True)
            right_text = right.astype("string").fillna("<NA>").reset_index(drop=True)
            if not left_text.equals(right_text):
                return False
    return True


def save_new_or_equivalent(frame, file_path):
    if file_path.is_file():
        existing = pd.read_csv(file_path, low_memory=False)
        if not frames_equivalent(frame.reset_index(drop=True), existing):
            raise FileExistsError(
                f"Existing artifact truly differs and was not overwritten: {file_path.name}"
            )
        return "already equivalent"
    frame.to_csv(file_path, index=False)
    return "created"


artifacts = {
    "model_scope_manifest.csv": model_scope,
    "family_screen_inner_fold_results.csv": family_folds,
    "family_screen_partition_summary.csv": family_summary,
    "retained_for_focused_tuning.csv": retained_combinations,
    "retained_model_family_frequency.csv": family_frequency,
    "dummy_benchmark_inner_fold_results.csv": dummy_folds,
    "dummy_benchmark_partition_summary.csv": dummy_summary,
    "model_family_screening_validation.csv": validation,
}

save_rows = []
for filename, frame in artifacts.items():
    status = save_new_or_equivalent(frame, output_directory / filename)
    save_rows.append({"artifact": filename, "status": status, "rows": len(frame)})

save_summary = pd.DataFrame(save_rows)
display(save_summary)
print("Notebook 07 can start from retained_for_focused_tuning.csv.")


,artifact,status,rows
0,model_scope_manifest.csv,created,10
1,family_screen_inner_fold_results.csv,created,5400
2,family_screen_partition_summary.csv,created,1080
3,retained_for_focused_tuning.csv,created,120
4,retained_model_family_frequency.csv,created,21
5,dummy_benchmark_inner_fold_results.csv,created,300
6,dummy_benchmark_partition_summary.csv,created,60
7,model_family_screening_validation.csv,created,11


Notebook 07 can start from retained_for_focused_tuning.csv.


## Interpretation boundary

Model-family frequencies summarize local inner-validation decisions. They do not define one global winning family and cannot be revised using outer-test performance.

Notebook 07 will tune only the two retained combinations within each seed and target, refit the local winner on the full outer-training set, and then score that seed's untouched outer test once.
